# Step 1: Install Required Libraries

In [13]:
!pip install pymupdf python-pptx pandas openpyxl

# Step 2: Upload Source File
# Input: DOCX-exported PDF / content PDF

In [14]:
from google.colab import files
uploaded = files.upload()

input_file = list(uploaded.keys())[0]
print(input_file)

Saving Assessment 2.docx.pdf to Assessment 2.docx (3).pdf
Assessment 2.docx (3).pdf


# Step 3: Extract Text from Source PDF
# Output: Full text used for MCQ and CQ extraction

In [15]:
import fitz  # PyMuPDF

pdf_doc = fitz.open(input_file)

pages_text = []

for page_num, page in enumerate(pdf_doc, start=1):
    page_text = page.get_text()
    pages_text.append(page_text)

text = "\n".join(pages_text)

print("Total pages:", len(pdf_doc))
print("Extracted characters:", len(text))
print(text[:1500])

Total pages: 3
Extracted characters: 3056
Assessment Round | 10 Minute School ​
Intern Recruitment  
 
MCQ: 
১। Energy currency ency শব্দটি শব্দটি ককোন ককোষ অঙ্গাণুর সাথে সম্পর্কিত? [ব. ববো. ২০১৯] 
(ক) গলগি বস্তু                              ​
(খ) মাইটটোকন্ড্রিয়ন        ​  
(গ) নিউক্লিয়াস                            ​
(ঘ) রাইববোসসোম  ​           
উত্তর: খ​ 
২। ককোষের শক্তি উৎপাদনকারী অঙ্গাণু ককোনটি? [সি. ববো ২০১৭] 
(ক) ক্লোররোপ্লাস্ট                          ​(খ) রাইববোজজোম             ​  
(গ) মাইটটোকন্ড্রিয়া                      ​ (ঘ) গলজি বস্তু     ​
           
উত্তর: গ​ 
৩। ক্রেবস চক্র ককোষের ককোন অঙ্গাণুতে সম্পন্ন হয়? [ঢা. ববো. ২০১৬] 
(ক) মাইটটোকন্ড্রিয়াতে                ​
(খ) নিউক্লিয়াসে              ​ 
(গ) গলজি বস্তুতে                        ​(ঘ) রাইববোজজোমে           ​  
 উত্তর: ক​
 
৪। অক্সিডেটিভ ফসফফোরাইলেশন সংঘটিত হয়- [য. ববো. ২০১৬] 
(ক) সাইটটোপ্লাজমে                     ​ (খ) রাইববোজজোমে           ​  
(গ) ক্লোররোপ্লাস্টে                        ​ (ঘ) মাইটটোকন্ড্রিয়নে      ​
 

# Step 4: Part A - Automated MCQ Extraction & Cleaning
# Output: Structured DataFrame with Correct Answer Marking

In [16]:
import re
import pandas as pd

def clean(value):
    value = str(value).replace("\u200b", "").replace("\xa0", " ")
    value = re.sub(r"\s+", " ", value)
    return value.strip()

def normalize_bangla_duplicates(value):
    # Fix some PDF extraction duplicate-letter issues
    fixes = {
        "কক": "ক",
        "ববো": "বো",
        "ররো": "রো",
        "টটো": "টো",
        "ববো": "বো",
        "জজো": "জো",
        "সসো": "সো",
        "মমো": "মো",
        "ক্রক্রো": "ক্রো",
    }
    value = str(value)
    for wrong, right in fixes.items():
        value = value.replace(wrong, right)
    return clean(value)

# Take only MCQ section
mcq_text = text.split("MCQ:")[-1].split("CQ:")[0]

# Split by answer, so each MCQ block stays separate
blocks = re.split(r"উত্তর\s*[:：]\s*([কখগঘ])", mcq_text)

mcq_rows = []

for i in range(0, len(blocks)-1, 2):
    block = blocks[i].strip()
    correct = blocks[i+1].strip()

    # Find question number
    q_match = re.search(r"([০-৯0-9]+)[।.]\s*(.*)", block, re.DOTALL)
    if not q_match:
        continue

    q_body = q_match.group(2).strip()

    # Extract question and options
    opt_match = re.search(
        r"(.*?)\(?ক\)\s*(.*?)\s*\(?খ\)\s*(.*?)\s*\(?গ\)\s*(.*?)\s*\(?ঘ\)\s*(.*)",
        q_body,
        re.DOTALL
    )

    if not opt_match:
        continue

    question = normalize_bangla_duplicates(opt_match.group(1))
    opt_a = normalize_bangla_duplicates(opt_match.group(2))
    opt_b = normalize_bangla_duplicates(opt_match.group(3))
    opt_c = normalize_bangla_duplicates(opt_match.group(4))
    opt_d = normalize_bangla_duplicates(opt_match.group(5))

    mcq_rows.append({
        "title": question,
        "options_1_answer": opt_a,
        "options_1_is_correct": 1 if correct == "ক" else 0,
        "options_2_answer": opt_b,
        "options_2_is_correct": 1 if correct == "খ" else 0,
        "options_3_answer": opt_c,
        "options_3_is_correct": 1 if correct == "গ" else 0,
        "options_4_answer": opt_d,
        "options_4_is_correct": 1 if correct == "ঘ" else 0,
    })

df = pd.DataFrame(mcq_rows)

print("Final MCQs:", len(df))
df

Final MCQs: 10


,title,options_1_answer,options_1_is_correct,options_2_answer,options_2_is_correct,options_3_answer,options_3_is_correct,options_4_answer,options_4_is_correct
0,Energy currency ency শব্দটি শব্দটি কোন কোষ অঙ্...,গলগি বস্তু,0,মাইটোকন্ড্রিয়ন,1,নিউক্লিয়াস,0,রাইবোসোম,0
1,কোষের শক্তি উৎপাদনকারী অঙ্গাণু কোনটি? [সি. বো ...,ক্লোরোপ্লাস্ট,0,রাইবোজোম,0,মাইটোকন্ড্রিয়া,1,গলজি বস্তু,0
2,ক্রেবস চক্র কোষের কোন অঙ্গাণুতে সম্পন্ন হয়? [ঢ...,মাইটোকন্ড্রিয়াতে,1,নিউক্লিয়াসে,0,গলজি বস্তুতে,0,রাইবোজোমে,0
3,অক্সিডেটিভ ফসফফোরাইলেশন সংঘটিত হয়- [য. বো. ২০১৬],সাইটোপ্লাজমে,0,রাইবোজোমে,0,ক্লোরোপ্লাস্টে,0,মাইটোকন্ড্রিয়নে,1
4,নিচের কোনটিতে কোয়ান্টোসোম পাওয়া যায়? [সি. বো. ...,রাইবোসোম,0,গলগি বস্তু,0,ক্লোরোপ্লাস্ট,1,মাইটোকন্ড্রিয়ন,0
5,থাইলাকয়েড কোষের কোন অঙ্গাণুতে থাকে? [ঢা. বো. ২...,মাইটোকন্ড্রিয়ায়,0,রাইবোজোমে,0,ক্লোরোপ্লাস্টে,1,লাইসোজোমে,0
6,শক্তি রূপান্তরের সাথে জড়িত অঙ্গাণু কোনটি? [কু....,রাইবোজোম,0,ক্লোরোপ্লাস্ট,1,লাইসোজোম,0,গলজি বস্তু,0
7,নিউক্লিয়াসের উপাদান কোনটি?[ঢা. বো. ২০১৬],ক্রোমোসোম,1,লাইসোজোম,0,রাইবোজোম,0,সেট্রোজোম,0
8,"ATC যদি DNA-এর অনুক্রম হয়, তাহলে উৎপন্ন mRNA- ...",TAG,1,UAG,0,UUG,0,TAC,0
9,দ্বিসূত্রক নিউক্লিক অ্যাসিডের নাইট্রোজেনঘটিত ক...,ATGC,0,CAGT,0,GACT,1,TGAC,0


# Step 4.1: Export MCQ Data to CSV and XLSX

In [17]:
df.to_csv("mcq_output.csv", index=False, encoding="utf-8-sig")
df.to_excel("mcq_output.xlsx", index=False)

files.download("mcq_output.csv")
#files.download("mcq_output.xlsx")

print("CSV files generated successfully.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CSV and XLSX files generated successfully.


# Step 5: Part B - Generate PPTX Using Provided Template
# Output: Presentation with MCQ and CQ Slides

In [18]:
from google.colab import files
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
import re

template_upload = files.upload()
template_file = list(template_upload.keys())[0]

prs = Presentation(template_file)

while len(prs.slides) > 0:
    rId = prs.slides._sldIdLst[0].rId
    prs.part.drop_rel(rId)
    del prs.slides._sldIdLst[0]

WHITE = RGBColor(255, 255, 255)
GREEN = RGBColor(20, 120, 75)
DARK = RGBColor(5, 28, 19)
LIGHT_GREEN = RGBColor(190, 255, 210)
YELLOW = RGBColor(255, 224, 102)

def add_box(slide, x, y, w, h, color=DARK, line_color=GREEN):
    box = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(x), Inches(y), Inches(w), Inches(h)
    )
    box.fill.solid()
    box.fill.fore_color.rgb = color
    box.line.color.rgb = line_color
    box.line.width = Pt(1.5)
    return box

def add_text(slide, text, x, y, w, h, size=20, bold=False, color=WHITE):
    box = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = box.text_frame
    tf.clear()
    tf.word_wrap = True

    p = tf.paragraphs[0]
    p.text = text
    p.font.name = "Noto Sans Bengali"
    p.font.size = Pt(size)
    p.font.bold = bold
    p.font.color.rgb = color
    return box

def add_footer(slide, number_text):
    add_text(slide, number_text, 0.65, 6.85, 2.5, 0.3, size=10, color=RGBColor(180, 220, 190))

def add_cover_slide():
    slide = prs.slides.add_slide(prs.slide_layouts[0])
    add_box(slide, 0.55, 0.75, 5.85, 4.85)
    add_text(slide, "Assessment Round\n10 Minute School", 0.9, 1.05, 5.2, 1.2, size=30, bold=True)
    add_text(slide, "Automated Question Extraction\n& Content Conversion", 0.9, 2.55, 5.1, 0.9, size=24, bold=True, color=LIGHT_GREEN)
    add_text(slide, "Generated automatically from the provided content file.", 0.9, 4.0, 5.1, 0.6, size=18)
    add_footer(slide, "Generated by Python Automation")

def add_mcq_slide(i, row):
    slide = prs.slides.add_slide(prs.slide_layouts[0])

    add_box(slide, 0.55, 0.6, 6.1, 5.95)

    add_text(slide, f"MCQ {i+1}", 0.9, 0.85, 2.5, 0.45, size=26, bold=True, color=YELLOW)

    question = str(row["title"]).strip()
    add_text(slide, question, 0.9, 1.45, 5.35, 0.95, size=20, bold=True)

    options = [
        ("ক", row["options_1_answer"], row["options_1_is_correct"]),
        ("খ", row["options_2_answer"], row["options_2_is_correct"]),
        ("গ", row["options_3_answer"], row["options_3_is_correct"]),
        ("ঘ", row["options_4_answer"], row["options_4_is_correct"]),
    ]

    y = 2.65
    for label, answer, correct in options:
        prefix = "✅" if int(correct) == 1 else "•"
        color = LIGHT_GREEN if int(correct) == 1 else WHITE
        add_text(slide, f"{prefix} {label}) {answer}", 1.05, y, 5.2, 0.45, size=19, bold=bool(int(correct)), color=color)
        y += 0.55

    correct_label = [label for label, answer, correct in options if int(correct) == 1][0]
    add_text(slide, f"Correct Answer: {correct_label}", 1.05, 5.35, 4.8, 0.45, size=18, bold=True, color=YELLOW)

    add_footer(slide, f"MCQ Slide {i+1}")

def split_text(content, limit=520):
    words = content.split()
    chunks, current = [], ""

    for word in words:
        if len(current) + len(word) + 1 <= limit:
            current += " " + word
        else:
            chunks.append(current.strip())
            current = word

    if current.strip():
        chunks.append(current.strip())

    return chunks

def add_cq_slide(title, content, footer):
    slide = prs.slides.add_slide(prs.slide_layouts[0])
    add_box(slide, 0.55, 0.6, 6.1, 5.95)

    add_text(slide, title, 0.9, 0.85, 4.8, 0.45, size=26, bold=True, color=YELLOW)

    lines = content.strip().split("\n")
    clean_content = "\n".join([line.strip() for line in lines if line.strip()])
    add_text(slide, clean_content, 0.9, 1.45, 5.45, 4.75, size=17)

    add_footer(slide, footer)

add_cover_slide()

for i, row in df.iterrows():
    add_mcq_slide(i, row)

cq_part = text.split("CQ:")[-1].strip()
cq_blocks = re.split(r'\n(?=প্রশ্ন:)', cq_part)

cq_no = 1
for cq in cq_blocks:
    if cq.strip():
        chunks = split_text(cq.strip(), limit=520)
        for part_no, chunk in enumerate(chunks, start=1):
            title = f"CQ {cq_no}" if len(chunks) == 1 else f"CQ {cq_no} | Part {part_no}"
            add_cq_slide(title, chunk, title)
        cq_no += 1

prs.save("professional_10ms_question_slides.pptx")
files.download("professional_10ms_question_slides.pptx")

Saving Assessment 2(b).pptx to Assessment 2(b) (5).pptx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>